# Lesson 1 - Models, Prompts and Parsers


## Querying without LangChain

Here we use the `openai` library directly to query the model hosted locally on `LMStudio`. `Gemma 3 12b` was used for this example at the time of writing.


In [2]:
from enum import StrEnum
from pprint import pprint

from openai import OpenAI


class LMStudio(StrEnum):
    BASE_URL = "http://localhost:1234/v1"
    API_KEY = "lm-studio"
    MODEL = "model-identifier"


def get_completion(prompt: str, model=LMStudio.MODEL) -> str | None:
    # create client
    client = OpenAI(base_url=LMStudio.BASE_URL, api_key=LMStudio.API_KEY)

    # from message as the user
    message = [{"role": "user", "content": prompt}]

    # query the llm
    response = client.chat.completions.create(
        model=model, messages=message, temperature=0
    )

    pprint(response.to_dict())
    return response.choices[0].message.content


get_completion("What is 1+1?")

{'choices': [{'finish_reason': 'stop',
              'index': 0,
              'logprobs': None,
              'message': {'content': '1 + 1 = 2\n', 'role': 'assistant'}}],
 'created': 1744842424,
 'id': 'chatcmpl-qi7p7glrffbblj5ndpbf',
 'model': 'gemma-3-12b-it',
 'object': 'chat.completion',
 'stats': {},
 'system_fingerprint': 'gemma-3-12b-it',
 'usage': {'completion_tokens': 8, 'prompt_tokens': 16, 'total_tokens': 24}}


'1 + 1 = 2\n'

### Some advanced prompting:


In [5]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse,\
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

style = """American English \
in a calm and respectful tone
"""

prompt = f"""Translate the text \
that is delimited by triple backticks 
into a style that is {style}.
text: ```{customer_email}```
"""

get_completion(prompt)

{'choices': [{'finish_reason': 'stop',
              'index': 0,
              'logprobs': None,
              'message': {'content': "Okay, here's a translation of the text "
                                     'into American English, maintaining a '
                                     'calm and respectful tone:\n'
                                     '\n'
                                     '"I’m quite frustrated – the lid flew off '
                                     'my blender and made a mess on my kitchen '
                                     'walls with smoothie! To top it off, the '
                                     "warranty doesn't cover the cost of "
                                     'cleaning up. I could really use some '
                                     'assistance right now."',
                          'role': 'assistant'}}],
 'created': 1744835572,
 'id': 'chatcmpl-uifmiaymapmifzbnh2jaqf',
 'model': 'gemma-3-12b-it',
 'object': 'chat.completion',
 'stats

'Okay, here\'s a translation of the text into American English, maintaining a calm and respectful tone:\n\n"I’m quite frustrated – the lid flew off my blender and made a mess on my kitchen walls with smoothie! To top it off, the warranty doesn\'t cover the cost of cleaning up. I could really use some assistance right now."'

## Querying WITH LangChain


In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain_core.messages.base import BaseMessage  # for typing
from langchain_openai import ChatOpenAI
from pydantic import SecretStr

# create a new client
client = ChatOpenAI(
    api_key=SecretStr(LMStudio.API_KEY),
    model=LMStudio.MODEL,
    base_url=LMStudio.BASE_URL,
    temperature=0,
)


# this is a prompt template string -> input variables are `style` and `text`
template_string = """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{text}```
"""
# initializing the prompt template
prompt_template = ChatPromptTemplate.from_template(template=template_string)

prompt_template.messages[0].prompt.input_variables  # prints the ['style', 'text']

# build a prompt using the prompt template:
customer_style = """American English \
in a calm and respectful tone
"""
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

customer_messages: list[BaseMessage] = prompt_template.format_messages(
    style=customer_style,
    text=customer_email,  # we input the input variable expected by the template
)


# call the LLM to translate to the style of the customer message
customer_response = client.invoke(customer_messages)

pprint(customer_response.content)

("Okay, here's a translation of the text into American English, maintaining a "
 'calm and respectful tone:\n'
 '\n'
 '"I’m quite frustrated – the lid flew off my blender and splattered smoothie '
 "all over my kitchen walls! To top it off, the warranty doesn't cover the "
 'cost of cleaning up the mess. I could really use some assistance right '
 'now."\n'
 '\n'
 '**Key changes made:**\n'
 '\n'
 '*   Replaced pirate slang ("Arrr," "be fuming," "me," "matey") with more '
 'standard American English phrasing.\n'
 '*   Softened the intensity of the frustration ("quite frustrated" instead of '
 '"fuming").\n'
 '*   Used more formal language ("To top it off" instead of "And to make '
 'matters worse").\n'
 '*   Replaced "need yer help" with a more polite request for assistance ("I '
 'could really use some assistance").')


### Reusing the prompt template to form a reply that turns message into a polite english pirate tone


In [ ]:
# Build response using new style and text
service_reply = """Hey there customer, \
the warranty does not cover \
cleaning expenses for your kitchen \
because it's your fault that \
you misused your blender \
by forgetting to put the lid on before \
starting the blender. \
Tough luck! See ya!
"""

service_style_pirate = """\
a polite tone \
that speaks in English Pirate\
"""

# build the new message by reusing old prompt template
service_messages = prompt_template.format_messages(
    style=service_style_pirate, text=service_reply
)

print(service_messages[0].content)

# send to LLM
service_response = client.invoke(service_messages)
print(service_response.content)

Translate the text that is delimited by triple backticks into a style that is a polite tone that speaks in English Pirate. text: ```Hey there customer, the warranty does not cover cleaning expenses for your kitchen because it's your fault that you misused your blender by forgetting to put the lid on before starting the blender. Tough luck! See ya!
```



## Output Parsers

This helps us turn a unstructured output into a structured one by defining a schema.


In [ ]:
# Prompt template that helps us extract structured data from a customer review:
customer_review = """\
This leaf blower is pretty amazing.  It has four settings:\
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn. \
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
"""

review_template = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product \
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""

prompt_template = ChatPromptTemplate.from_template(review_template)

# build the message
message = prompt_template.format_messages(text=customer_review)

response = client.invoke(message)

pprint(response.content)  # this will be a string

('```json\n'
 '{\n'
 '  "gift": true,\n'
 '  "delivery_days": 2,\n'
 '  "price_value": [\n'
 '    "It\'s slightly more expensive than the other leaf blowers out there",\n'
 '    "but I think it\'s worth it for the extra features"\n'
 '  ]\n'
 '}\n'
 '```')


In [ ]:
### Defining a schema to deserialize the response

In [ ]:
from langchain.output_parsers import ResponseSchema, StructuredOutputParser

# create a schema to extract data from chat response
gift_schema = ResponseSchema(
    name="gift",
    description="Was the item purchased\
                as a gift for someone else? \
                Answer True if yes,\
                False if not or unknown.",
)
delivery_days_schema = ResponseSchema(
    name="delivery_days",
    description="How many days\
                did it take for the product\
                to arrive? If this \
                information is not found,\
                output -1.",
)
price_value_schema = ResponseSchema(
    name="price_value",
    description="Extract any\
                sentences about the value or \
                price, and output them as a \
                comma separated Python list.",
)

response_schemas = [gift_schema, delivery_days_schema, price_value_schema]


# using a new template -> extract text
review_template_2 = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product\
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

text: {text}

{format_instructions}
"""

# build the output parser & format instructions
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

prompt = ChatPromptTemplate.from_template(template=review_template_2)
messages = prompt.format_messages(
    text=customer_review, format_instructions=format_instructions
)  # pass format instructions into LLM

# send the prompt
response = client.invoke(input=messages)

# parse the prompt
output_dict: dict = output_parser.parse(str(response.content))

(output_dict)

{'gift': 'True',
 'delivery_days': '2',
 'price_value': ["It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."]}